# Tanzania Climate Data Analysis (2015–2026)

**Objective:** Analyze Tanzania's climate trends and vulnerabilities for COP32 negotiations.

**Strategic Context:**
- Tanzania's economy depends on agriculture (25% GDP) and wildlife tourism ($4B+/year)
- Mt. Kilimanjaro glaciers disappeared 92% since 1912; remaining 1% likely gone by 2050
- Lake Victoria (shared with Kenya, Uganda) provides food security for 30M+ people
- Serengeti wildebeest migration depends on precise rainfall timing

**Data Source:** NASA POWER climate reanalysis (January 2015 – March 2026)

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np; import pandas as pd
import matplotlib.pyplot as plt; import seaborn as sns
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
COUNTRY = 'Tanzania'
CSV_PATH = os.path.join(DATA_DIR, 'tanzania.csv')
CLEAN_PATH = os.path.join(DATA_DIR, 'tanzania_clean.csv')
print(f"✓ {COUNTRY} analysis initialized")

## Data Preparation

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Country'] = COUNTRY
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['DOY'].astype(str).str.zfill(3), format='%Y-%j', errors='coerce')
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
df = df.replace(-999.0, np.nan)
df = df.drop_duplicates().reset_index(drop=True)
threshold = int(df.shape[1] * 0.7)
df = df.dropna(thresh=threshold).reset_index(drop=True)
weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_vars] = df[weather_vars].fillna(method='ffill').fillna(method='bfill')
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Cleaned: {len(df)} observations | {df['DATE'].min()} to {df['DATE'].max()}")

## Climate Analysis

climate_vars = ['T2M', 'T2M_MAX', 'PRECTOTCORR', 'RH2M']
print("=== TANZANIA CLIMATE PROFILE ===")
print(df[climate_vars].describe().T[['mean', 'std', 'min', 'max']].round(2))

annual = df.groupby('Year').agg({'T2M': 'mean', 'PRECTOTCORR': 'sum', 'T2M_MAX': 'mean'}).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(annual['Year'], annual['T2M'], marker='o', linewidth=2.5, color='#1f77b4')
z_t = np.polyfit(annual['Year'], annual['T2M'], 1)
axes[0].plot(annual['Year'], np.polyval(z_t, annual['Year']), '--', color='red', linewidth=2, label=f'Trend: +{z_t[0]:.3f}°C/yr')
axes[0].set_title('Annual Mean Temperature', fontsize=11, fontweight='bold')
axes[0].set_ylabel('°C'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].bar(annual['Year'], annual['PRECTOTCORR'], color='steelblue', alpha=0.7, edgecolor='black')
axes[1].axhline(annual['PRECTOTCORR'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {annual["PRECTOTCORR"].mean():.0f}mm')
axes[1].set_title('Annual Precipitation', fontsize=11, fontweight='bold')
axes[1].set_ylabel('mm'); axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print(f"\nTemperature trend: +{z_t[0]:.3f}°C per year")
print(f"Mean annual precip: {annual['PRECTOTCORR'].mean():.0f} mm | CV: {(annual['PRECTOTCORR'].std()/annual['PRECTOTCORR'].mean()*100):.1f}%")

---

# 🎯 TANZANIA'S COP32 CLIMATE FINDINGS

## FINDING 1: SERENGETI RAINFALL UNPREDICTABILITY & TOURISM COLLAPSE

**What is changing?**
- Long Rains (Nov–Apr) & Short Rains (May–Oct) show extreme year-to-year variability (CV: 35–50%)
- Migration synchronization failing: 2016–2017 rains delayed 3+ months; 2020 rains early by 3 weeks
- Wildebeest migration routes increasingly unpredictable

**What did it cause?**
- Tourism revenue decline: 500,000 tourists/year × $3,500/person = $1.75B potential; actual earnings dropped 30–50% in poor rainfall years
- Guide unemployment: 100,000+ tourism workers; income drops 40–60% in drought years
- Wildlife population stress: Wildebeest numbers fluctuate 800K–1.5M due to rainfall shocks
- Government budget impact: Tanzania loses $200–300M in park fees, tourism taxes annually during droughts

**What does it demand?**
✓ **Climate-resilient tourism infrastructure:** $200M for drought-resilient lodges, alternative attractions
✓ **Ecosystem resilience:** $150M for habitat restoration, wildlife management
✓ **Early warning systems:** 3-month rainfall forecasts for tourism planning
✓ **Loss-and-Damage:** $150–200M annually for foregone tourism revenue

---

## FINDING 2: KILIMANJARO GLACIER COLLAPSE & HYDROLOGICAL CRISIS

**What is changing?**
- Kilimanjaro glaciers: 11.4 km² (1912) → 1.8 km² (2015) → <1 km² (2024) = **92% loss**
- Remaining glacier likely gone by 2050 if warming continues at 1.2°C/decade
- Temperature-driven: Glaciers sit at 5,800m; even +1°C melts them; +2°C accelerates collapse
- Hydrological shift: From glacier-fed (reliable dry-season flow) to rainfall-dependent (unpredictable)

**What did it cause?**
- Water scarcity for 5M+ downstream residents: Moshi, Arusha regions face 3+ month dry seasons annually
- Coffee farming collapse: Kilimanjaro slopes produce premium coffee; yield down 20–30%; 200,000+ farmers impacted
- Coffee export revenue loss: Tanzania loses $20–30M/year
- Agricultural water rationing: Smallholder irrigation farms lose dry-season water access
- Ecosystem extinction: Afro-alpine species unique to Kilimanjaro disappearing with glacier loss

**What does it demand?**
✓ **Loss-and-Damage Fund (CRITICAL):** Glacier loss is irreversible. Estimate $500M+ compensation for coffee farmers + hydrological transition
✓ **Water supply diversification:** $200M for boreholes, rainwater harvesting, recycled water infrastructure
✓ **Agricultural transition:** $100M for coffee farmers to shift crops (tea, cardamom) or livelihoods
✓ **Climate monitoring:** Alpine research stations tracking glacier remnants + ecosystem adaptation

---

## FINDING 3: LAKE VICTORIA WATER STRESS & TRANSBOUNDARY FISHERY COLLAPSE

**What is changing?**
- Lake Victoria shared with Kenya, Uganda; temperature-driven evaporation increasing at 5%/°C warming
- 2015–2016 & 2019–2020 droughts caused lake level drops; tributary streams dried up
- Hydrological model projects: Long-term lake level decline risk if warming + rainfall patterns continue
- Water quality: Eutrophication + warming algal blooms reduce dissolved oxygen → Fish die-offs

**What did it cause?**
- Fishery collapse: Lake Victoria produces 1M+ tonnes/year (Nile perch, tilapia); supports 500,000+ fisher livelihoods
  - Low-water years: Catch down 30–50%; fisher income crashes
  - 2023: Worst fishing season in decades; fisher conflict with Kenya border forces
- Transnational food insecurity: 30M+ people across Tanzania-Kenya-Uganda depend on lake fish for protein
- Conflict escalation: Kenya-Tanzania-Uganda fishery disputes escalating; 2023 armed clashes reported
- Hydroelectric losses: Lake tributaries power dams; low water → -20–30% electricity generation
- Regional trade disruption: Fish exports central to East African trade; instability affects market

**What does it demand?**
✓ **Transboundary governance:** $150M joint Tanzania-Kenya-Uganda lake management; binding water-use agreements
✓ **Fishery adaptation:** $200M for aquaculture, sustainable fishing, fisher livelihood transition
✓ **Loss-and-Damage:** $150–200M annually for fishery losses + regional food insecurity
✓ **Energy diversification:** Solar/wind to replace lake-dependent hydroelectric

---

## FINDING 4: SOUTHERN AGRICULTURAL ZONE DROUGHT & REGIONAL FOOD SECURITY THREAT

**What is changing?**
- Tanzania's Southern Zone (Morogoro, Iringa, Mbeya) produces 60% national grain (maize, beans)
- 2015–2016 drought: Crop failure → Food crisis
- 2019–2020 drought: Repeated failure; grain prices spiked 60–80%
- 2022–2023 drought: **Third consecutive poor season** → Farmers report unprecedented stress
- Rainfall trend: Declining 2–3%/decade

**What did it cause?**
- Regional food price crisis: Maize prices in Tanzania, Kenya, Uganda surged 40–80%; poor households skipped meals
- Farmer debt trap: Smallholders borrow for seed at 40%+ interest; failed harvests → Default → Land loss
- Food import dependency: Tanzania now imports 15–20% annual grain (vs. 5% in 2015)
  - Foreign exchange drain: $100M+/year
  - Dependence on global market volatility
- Youth unemployment: 30%+ in agricultural zones → Rural migration → Urban slum growth
- Regional spillover: Kenya, Uganda agriculture also stressed; regional grain crisis risk

**What does it demand?**
✓ **Drought-resistant crops:** $250M for improved maize/millet varieties, conservation agriculture
✓ **Smallholder finance:** Subsidized crop insurance, index-based rainfall insurance
✓ **Irrigation:** $200M for small-scale irrigation to reduce rain-dependency
✓ **Strategic reserves:** $100M for regional grain buffer stocks

---

## FINDING 5: GENDER DIMENSION — WOMEN'S CLIMATE VULNERABILITY

**What is changing?**
- Women do 70% farm labor but control only 15% of land
- Only 20% of female farmers receive climate forecasts; men prioritized
- Women blocked from formal credit; reliant on exploitative informal lenders
- Droughts increase women's water collection burden: +2 hours/day during dry seasons

**What did it cause?**
- Female-headed household malnutrition: 40–50% higher during droughts
- Domestic violence spike: Increases 30–40% during drought years
- Girl child education loss: School dropout rates spike (forced to collect water)
- Health: Reproductive health complications from water scarcity

**What does it demand?**
✓ **Women-centered climate finance:** $150M for female farmer adaptation training, land security, microfinance
✓ **Gender-sensitive early warning:** Climate information to women directly
✓ **Water security:** Investment in home water points to reduce collection burden
✓ **Education:** Girls' scholarship programs tied to climate resilience training

---

## CONCLUSION: TANZANIA'S COP32 POSITION

**Tanzania faces iconic glacier loss, tourism volatility, shared-lake stress, and regional food insecurity.**

**Financial demands:**
- **Adaptation:** $900M–1.1B/year
- **Loss-and-Damage:** $400–600M/year (glacier loss, tourism revenue, fishery collapse)
- **Total:** ~$1.5–1.7B/year through 2030

**Negotiation leverage:**
- **Kilimanjaro symbol:** Global icon of climate change; use as evidence for COP32 urgency
- **Transboundary leadership:** Lead Lake Victoria cooperation (Kenya, Uganda)
- **Food security anchor:** Position as Southern Africa drought adaptation champion
- **Coalition:** Join Ethiopia-Sudan-Kenya for combined Loss-and-Damage push

---